<a href="https://colab.research.google.com/github/davidriveraarbelaez/Optativa3_Agentes_IA/blob/main/Sistemas_multiagentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Fundamentos de colaboración y comunicación**

**Teoría:** En un MAS, la colaboración requiere un mecanismo para compartir información. En LangGraph, esto se logra a través de un estado compartido (State), un diccionario central que todos los nodos (agentes) leen y actualizan. El flujo se define en un grafo, conectando nodos con aristas, lo que permite pasar de la orquestación lineal a flujos condicionales y cíclicos.

**Práctica (Dos Agentes con LangGraph):** Implementarás un sistema simple de Escritor y Crítico.

In [ ]:
%%capture
!pip install langchain_google_genai
!pip install langgraph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# 1. DEFINIR EL ESTADO COMPARTIDO (Teoría aplicada)
class EscrituraState(TypedDict):
    tema: str
    primer_borrador: str
    critica: str
    borrador_final: str

In [ ]:
import os
# 2. INICIALIZAR AGENTES/LLMs
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
# 3. DEFINIR LOS NODOS (Funciones que actúan sobre el estado)
def nodo_escritor(state: EscrituraState) -> EscrituraState:
    """Agente especializado en escribir."""
    prompt = f"Escribe un párrafo breve sobre: {state['tema']}"
    respuesta = llm.invoke(prompt)
    return {"primer_borrador": respuesta.content}

def nodo_critico(state: EscrituraState) -> EscrituraState:
    """Agente especializado en revisar."""
    prompt = f"Revisa este texto y sugiere una mejora concisa: {state['primer_borrador']}"
    respuesta = llm.invoke(prompt)
    return {"critica": respuesta.content}

def nodo_refinador(state: EscrituraState) -> EscrituraState:
    """Agente que integra la crítica."""
    prompt = f"Basándote en el borrador '{state['primer_borrador']}' y la crítica '{state['critica']}', escribe la versión final."
    respuesta = llm.invoke(prompt)
    return {"borrador_final": respuesta.content}


In [ ]:
# 4. CONSTRUIR EL GRAFO DE FLUJO DE TRABAJO
grafo = StateGraph(EscrituraState)
grafo.add_node("escritor", nodo_escritor)
grafo.add_node("critico", nodo_critico)
grafo.add_node("refinador", nodo_refinador)

# Graficar el nodo

In [ ]:
# 5. DEFINIR EL FLUJO (Aristas)
grafo.add_edge(START, "escritor")
grafo.add_edge("escritor", "critico")
grafo.add_edge("critico", "refinador")
grafo.add_edge("refinador", END)

In [ ]:
# 6. COMPILAR Y EJECUTAR
app = grafo.compile()
estado_inicial = {"tema": "el futuro de la energía renovable"}
resultado = app.invoke(estado_inicial)
print("Borrador Final:", resultado["borrador_final"])

Borrador Final: Considerando el borrador, la crítica y la sugerencia de mejora concisa para simplificar la estructura y eliminar redundancias, aquí tienes la versión final:

---

**Versión Final:**

La energía renovable se consolida como la columna vertebral del sistema energético global. La constante reducción de costos, los avances en almacenamiento y eficiencia, y la creciente urgencia climática impulsarán una integración más profunda y descentralizada de fuentes como la solar, eólica, geotérmica e hidroeléctrica. Esto no solo garantizará mayor independencia y seguridad energética, sino que será crucial para descarbonizar la economía global, construir un futuro sostenible y desarrollar redes más inteligentes y resilientes.
